# Fase 1 — GSE54888 (Cat. B: Validazione DGF vs non-DGF)

## Dataset
| Campo | Valore |
|-------|--------|
| **GEO** | GSE54888 |
| **Piattaforma** | GPL6244 — Affymetrix HuGene-1.0-ST (gene-level) |
| **Campioni** | 54 biopsie pre-impianto da donatori deceduti |
| **Preprocessing GEO** | RMA (log2 + quantile-normalized) |
| **Confronto** | DGF (n=27) vs non-DGF (n=27) |
| **Sottogruppi** | pDGF >14gg (n=13), eGFR Low (n=18) vs High (n=35) |
| **Categoria** | **B — Validazione severità** |
| **Articolo** | Mourão et al., Hum Immunol 2016; 77(4):353-7 |

## Vantaggi di questo dataset
- Piattaforma **gene-level** (GPL6244): ogni riga = un gene, NO aggregazione probe
- Metadati puliti: DGF occurrence, pDGF, eGFR — tutto nel series matrix
- Design bilanciato: 27 DGF vs 27 non-DGF
- Geni validati per verifica: DEFB1, FABP3, GK (DOWN in DGF), da Mourão 2016

### Pipeline:
1. Metadata dal series matrix
2. Caricamento matrice espressione (RMA-preprocessed)
3. QC
4. Annotazione transcript cluster → gene (GPL6244)
5. Definizione gruppi
6. Verifica geni dall'articolo (DEFB1, FABP3, GK)
7. Espressione differenziale (Welch t-test + FDR)
8. Salvataggio output

In [1]:
# Setup
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "geo" / "GSE54888"
OUTPUT_DIR   = PROJECT_ROOT / "output" / "fase1" / "GSE54888"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERIES_MAT_FILE = DATA_DIR / "GSE54888_series_matrix.txt.gz"

FDR_THRESHOLD = 0.05
FC_THRESHOLD  = 0.379  # |log2FC| > 0.379 → FC ≥ 1.3

print(f"Project root:  {PROJECT_ROOT}")
print(f"Data dir:      {DATA_DIR}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Series matrix: {SERIES_MAT_FILE}")
print(f"File exists:   {SERIES_MAT_FILE.exists()}")

Project root:  /Users/robertocasale/Desktop/lavoro_FBF/magic_solution
Data dir:      /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/data/geo/GSE54888
Output dir:    /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/output/fase1/GSE54888
Series matrix: /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/data/geo/GSE54888/GSE54888_series_matrix.txt.gz
File exists:   True


## 1. Metadata dal Series Matrix

In [2]:
# Parsing header del series matrix
sample_ids = []
sample_titles = []
source_names = []
sample_chars_raw = []

with pd.io.common.get_handle(SERIES_MAT_FILE, mode="r", compression="gzip") as f:
    for line in f.handle:
        line = line.strip()
        if line.startswith("!Sample_geo_accession"):
            sample_ids = [v.strip('"') for v in line.split("\t")[1:]]
        elif line.startswith("!Sample_title"):
            sample_titles = [v.strip('"') for v in line.split("\t")[1:]]
        elif line.startswith("!Sample_source_name_ch1"):
            source_names = [v.strip('"') for v in line.split("\t")[1:]]
        elif line.startswith("!Sample_characteristics_ch1"):
            row = [v.strip('"') for v in line.split("\t")[1:]]
            sample_chars_raw.append(row)
        elif line.startswith("!series_matrix_table_begin"):
            break

print(f"Campioni trovati:      {len(sample_ids)}")
print(f"Righe characteristics: {len(sample_chars_raw)}")
print(f"Primi 5 titoli:        {sample_titles[:5]}")

Campioni trovati:      54
Righe characteristics: 5
Primi 5 titoli:        ['TRIB-11', 'TRIB-14', 'TRIB-2', 'TRIB-7', 'TRIB-12']


In [3]:
# Costruisci metadata con matching ESATTO delle chiavi
meta = pd.DataFrame({
    "title": sample_titles,
    "source": source_names,
}, index=sample_ids)
meta.index.name = "gsm_id"

for row_idx, row in enumerate(sample_chars_raw):
    for sample_idx, val in enumerate(row):
        if ":" not in val:
            continue
        key, value = val.split(":", 1)
        key = key.strip().lower()
        value = value.strip()

        if key == "donor":
            meta.loc[sample_ids[sample_idx], "donor_id"] = value
        elif key == "tissue type":
            meta.loc[sample_ids[sample_idx], "tissue"] = value
        elif key == "dgf occurrence":
            meta.loc[sample_ids[sample_idx], "dgf"] = value
        elif key == "dgf lasting more than 14 days":
            meta.loc[sample_ids[sample_idx], "pdgf_14d"] = value
        elif key == "egfr":
            meta.loc[sample_ids[sample_idx], "egfr_group"] = value

print("Metadata shape:", meta.shape)
print("\nColonne:", meta.columns.tolist())
display(meta.head(10))
print()
display(meta.tail(5))

Metadata shape: (54, 7)

Colonne: ['title', 'source', 'donor_id', 'tissue', 'dgf', 'pdgf_14d', 'egfr_group']


,title,source,donor_id,tissue,dgf,pdgf_14d,egfr_group
gsm_id,,,,,,,
GSM1326154,TRIB-11,preimplantation kidney biopsy,11,preimplantation kidney biopsy,DGF,yes,High
GSM1326155,TRIB-14,preimplantation kidney biopsy,14,preimplantation kidney biopsy,DGF,no,High
GSM1326156,TRIB-2,preimplantation kidney biopsy,2,preimplantation kidney biopsy,DGF,no,Low
GSM1326157,TRIB-7,preimplantation kidney biopsy,7,preimplantation kidney biopsy,without DGF,--,High
GSM1326158,TRIB-12,preimplantation kidney biopsy,12,preimplantation kidney biopsy,DGF,no,High
GSM1326159,TRIB-132,preimplantation kidney biopsy,132,preimplantation kidney biopsy,without DGF,--,High
GSM1326160,TRIB-136,preimplantation kidney biopsy,136,preimplantation kidney biopsy,DGF,no,High
GSM1326161,TRIB-13,preimplantation kidney biopsy,13,preimplantation kidney biopsy,DGF,yes,High
GSM1326162,TRIB-16,preimplantation kidney biopsy,16,preimplantation kidney biopsy,without DGF,--,High


,title,source,donor_id,tissue,dgf,pdgf_14d,egfr_group
gsm_id,,,,,,,
GSM1326203,TRIB-173,preimplantation kidney biopsy,173,preimplantation kidney biopsy,without DGF,--,Low
GSM1326204,TRIB-19,preimplantation kidney biopsy,19,preimplantation kidney biopsy,DGF,yes,Low
GSM1326205,TRIB-293,preimplantation kidney biopsy,293,preimplantation kidney biopsy,without DGF,--,Low
GSM1326206,TRIB-294,preimplantation kidney biopsy,294,preimplantation kidney biopsy,without DGF,--,High
GSM1326207,TRIB-76,preimplantation kidney biopsy,76,preimplantation kidney biopsy,DGF,no,High


In [4]:
# Verifica gruppi vs descrizione GEO e articolo Mourão 2016
print("=" * 60)
print("VERIFICA GRUPPI")
print("=" * 60)

# DGF
print("\nDGF occurrence:")
print(meta["dgf"].value_counts())
n_dgf = (meta["dgf"] == "DGF").sum()
n_nodgf = (meta["dgf"] == "without DGF").sum()
print(f"\n  Atteso (GEO): DGF=27, without DGF=27")
print(f"  Trovato:      DGF={n_dgf}, without DGF={n_nodgf}")
assert n_dgf == 27, f"Attesi 27 DGF, trovati {n_dgf}"
assert n_nodgf == 27, f"Attesi 27 non-DGF, trovati {n_nodgf}"
print("  ✅ Confermato!")

# pDGF (>14 days)
print("\npDGF (>14 giorni):")
print(meta["pdgf_14d"].value_counts())
n_pdgf = (meta["pdgf_14d"] == "yes").sum()
n_npdgf = (meta["pdgf_14d"] == "no").sum()
print(f"\n  Atteso (GEO): pDGF=13, non-pDGF={27-13}=14")
print(f"  Trovato:      pDGF={n_pdgf}, non-pDGF={n_npdgf}")
assert n_pdgf == 13, f"Attesi 13 pDGF, trovati {n_pdgf}"
print("  ✅ Confermato!")

# eGFR
print("\neGFR group (1 anno post-Tx):")
print(meta["egfr_group"].value_counts())
n_high = (meta["egfr_group"] == "High").sum()
n_low = (meta["egfr_group"] == "Low").sum()
n_na = (meta["egfr_group"] == "--").sum()
print(f"\n  Atteso (GEO): High=35, Low=18 (1 mancante)")
print(f"  Trovato:      High={n_high}, Low={n_low}, NA={n_na}")
assert n_high == 35 and n_low == 18
print("  ✅ Confermato!")

# Tissue
print("\nTissue:")
print(meta["tissue"].value_counts())

VERIFICA GRUPPI

DGF occurrence:
dgf
DGF            27
without DGF    27
Name: count, dtype: int64

  Atteso (GEO): DGF=27, without DGF=27
  Trovato:      DGF=27, without DGF=27
  ✅ Confermato!

pDGF (>14 giorni):
pdgf_14d
--     27
no     14
yes    13
Name: count, dtype: int64

  Atteso (GEO): pDGF=13, non-pDGF=14=14
  Trovato:      pDGF=13, non-pDGF=14
  ✅ Confermato!

eGFR group (1 anno post-Tx):
egfr_group
High    35
Low     18
--       1
Name: count, dtype: int64

  Atteso (GEO): High=35, Low=18 (1 mancante)
  Trovato:      High=35, Low=18, NA=1
  ✅ Confermato!

Tissue:
tissue
preimplantation kidney biopsy    54
Name: count, dtype: int64


## 2. Caricamento Matrice Espressione (RMA)

GPL6244 è un array **gene-level** (HuGene-1.0-ST). A differenza di GPL570:
- Ogni riga è un **transcript cluster** (≈ un gene)
- NON servono aggregazioni probe → gene
- Gli ID sono numerici (es. 7892501, 7892502, ...)
- Il mapping ID → gene symbol viene dall'annotazione GPL6244

In [5]:
# Carica matrice espressione
print("Caricamento matrice espressione...")
skip = 0
with pd.io.common.get_handle(SERIES_MAT_FILE, mode="r", compression="gzip") as f:
    for i, line in enumerate(f.handle):
        if line.startswith("!series_matrix_table_begin"):
            skip = i + 1
            break

expr_raw = pd.read_csv(
    SERIES_MAT_FILE, sep="\t", compression="gzip",
    skiprows=skip, index_col=0, comment="!"
)
if expr_raw.index[-1] == "!series_matrix_table_end":
    expr_raw = expr_raw.iloc[:-1]
expr_raw = expr_raw.apply(pd.to_numeric, errors="coerce")
expr_raw.index = expr_raw.index.astype(str)

print(f"\nMatrice: {expr_raw.shape[0]} transcript clusters × {expr_raw.shape[1]} campioni")
print(f"Range valori: [{expr_raw.values.min():.2f}, {expr_raw.values.max():.2f}]")
print(f"Mediana: {np.nanmedian(expr_raw.values):.2f}")
print(f"→ Confermato log2 (RMA preprocessed)")
print(f"\nPrimi 5 ID (transcript clusters): {expr_raw.index[:5].tolist()}")
display(expr_raw.iloc[:5, :5])

Caricamento matrice espressione...



Matrice: 33252 transcript clusters × 54 campioni
Range valori: [1.12, 14.06]
Mediana: 6.49
→ Confermato log2 (RMA preprocessed)

Primi 5 ID (transcript clusters): ['7892501', '7892502', '7892503', '7892504', '7892505']


,GSM1326154,GSM1326155,GSM1326156,GSM1326157,GSM1326158
ID_REF,,,,,
7892501,5.367374,4.845314,4.142103,3.470416,3.458709
7892502,5.102689,5.020831,5.792261,5.446037,6.205698
7892503,4.674776,4.940126,4.126672,4.838853,3.377139
7892504,8.761185,8.732883,8.098607,8.603211,8.595263
7892505,4.201547,4.377568,4.548349,5.133498,4.662649


## 3. QC

In [6]:
medians = expr_raw.median()
print(f"Mediane per campione: mean={medians.mean():.4f}, SD={medians.std():.4f}")
if medians.std() > 0.5:
    print("⚠️  Mediane variabili")
else:
    print("✅ Distribuzioni uniformi — RMA confermata")
print(f"NaN: {expr_raw.isna().sum().sum()}")
iqrs = expr_raw.quantile(0.75) - expr_raw.quantile(0.25)
print(f"IQR: mean={iqrs.mean():.3f}, SD={iqrs.std():.3f}")

Mediane per campione: mean=6.4934, SD=0.0282
✅ Distribuzioni uniformi — RMA confermata
NaN: 0
IQR: mean=2.981, SD=0.079


## 4. Annotazione Transcript Cluster → Gene (GPL6244)

GPL6244 (HuGene-1.0-ST) è un array gene-level: ogni transcript cluster ID
corrisponde a un gene. Serve il mapping ID numerico → gene symbol.

In [7]:
# === Annotazione GPL6244 ===
tc_to_gene = {}
annotation_source = "none"

# --- Approccio 1: GEOparse ---
try:
    import GEOparse
    print("Tentativo 1: GEOparse...")
    gpl = GEOparse.get_GEO("GPL6244", destdir=str(DATA_DIR), silent=True)
    # GPL6244 ha colonne diverse — cerchiamo gene_assignment o gene symbol
    cols = gpl.table.columns.tolist()
    print(f"  Colonne disponibili: {cols[:10]}...")
    
    # Su GPL6244 il gene symbol è spesso in "gene_assignment"
    id_col = "ID"
    gene_col = None
    for c in cols:
        if "gene_assignment" in c.lower():
            gene_col = c
            break
        elif c.lower() in ("gene symbol", "gene_symbol", "symbol"):
            gene_col = c
            break
    
    if gene_col:
        print(f"  Usando colonna: {gene_col}")
        for _, row in gpl.table.iterrows():
            tc_id = str(row[id_col]).strip()
            gene_raw = str(row[gene_col]).strip()
            if tc_id and gene_raw and gene_raw not in ("", "nan", "---", "NA"):
                # gene_assignment ha formato "NM_xxxx // GENE // desc // ..."
                if "//" in gene_raw:
                    parts = [p.strip() for p in gene_raw.split("//")]
                    if len(parts) >= 2:
                        symbol = parts[1].strip()
                        if symbol and symbol not in ("", "---"):
                            tc_to_gene[tc_id] = symbol
                else:
                    tc_to_gene[tc_id] = gene_raw.split(" ")[0]
        annotation_source = "GEOparse"
        print(f"  ✅ {len(tc_to_gene)} transcript cluster → gene mappings")
    else:
        print(f"  ❌ Colonna gene non trovata in: {cols}")
except Exception as e:
    print(f"  ❌ GEOparse fallito: {e}")

# --- Approccio 2: File locale ---
if not tc_to_gene:
    for fname in ["GPL6244_annotation.csv", "GPL6244.annot"]:
        local_f = DATA_DIR / fname
        if local_f.exists():
            print(f"\nTentativo 2: {fname}...")
            try:
                df_ann = pd.read_csv(local_f, sep=None, engine="python", comment="#")
                for c in df_ann.columns:
                    if "gene" in c.lower() and "assign" in c.lower():
                        # Parse gene_assignment
                        for _, row in df_ann.iterrows():
                            tc_id = str(row.iloc[0]).strip()
                            ga = str(row[c]).strip()
                            if "//" in ga:
                                parts = [p.strip() for p in ga.split("//")]
                                if len(parts) >= 2 and parts[1] not in ("", "---"):
                                    tc_to_gene[tc_id] = parts[1]
                        break
                if tc_to_gene:
                    annotation_source = f"local ({fname})"
                    print(f"  ✅ {len(tc_to_gene)} mappings")
                    break
            except Exception as e:
                print(f"  ❌ {e}")

if tc_to_gene:
    mapped = sum(1 for p in expr_raw.index if p in tc_to_gene)
    print(f"\n✅ Annotazione: {annotation_source}")
    print(f"   Copertura: {mapped}/{len(expr_raw)} ({mapped/len(expr_raw)*100:.1f}%)")
    # Verifica geni chiave
    for gene in ["FABP3", "DEFB1", "GK", "ACSL4", "CUBN"]:
        found = [k for k, v in tc_to_gene.items() if v == gene]
        print(f"   {gene}: {len(found)} transcript cluster(s)")
else:
    print("\n⚠️  Annotazione non disponibile")
    print("   Scarica GPL6244 annotation da GEO e riprova")

Tentativo 1: GEOparse...
  Colonne disponibili: ['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment']...
  Usando colonna: gene_assignment
  ✅ 25293 transcript cluster → gene mappings

✅ Annotazione: GEOparse
   Copertura: 25293/33252 (76.1%)
   FABP3: 1 transcript cluster(s)
   DEFB1: 1 transcript cluster(s)
   GK: 2 transcript cluster(s)
   ACSL4: 1 transcript cluster(s)
   CUBN: 1 transcript cluster(s)


## 5. Definizione gruppi

In [8]:
# Gruppi principali
dgf_samples = meta[meta["dgf"] == "DGF"].index.tolist()
nodgf_samples = meta[meta["dgf"] == "without DGF"].index.tolist()

# Sottogruppi
pdgf_samples = meta[meta["pdgf_14d"] == "yes"].index.tolist()  # pDGF >14d
sdgf_samples = meta[(meta["dgf"] == "DGF") & (meta["pdgf_14d"] == "no")].index.tolist()  # short DGF

egfr_low = meta[meta["egfr_group"] == "Low"].index.tolist()
egfr_high = meta[meta["egfr_group"] == "High"].index.tolist()

print("GRUPPI:")
print(f"  DGF:          {len(dgf_samples)}")
print(f"  non-DGF:      {len(nodgf_samples)}")
print(f"  pDGF (>14d):  {len(pdgf_samples)}")
print(f"  short DGF:    {len(sdgf_samples)}")
print(f"  eGFR Low:     {len(egfr_low)}")
print(f"  eGFR High:    {len(egfr_high)}")

# Cross-tab DGF × eGFR
print("\nCross-tab DGF × eGFR:")
ct = pd.crosstab(meta["dgf"], meta["egfr_group"])
display(ct)

GRUPPI:
  DGF:          27
  non-DGF:      27
  pDGF (>14d):  13
  short DGF:    14
  eGFR Low:     18
  eGFR High:    35

Cross-tab DGF × eGFR:


egfr_group,--,High,Low
dgf,,,
DGF,1,17,9
without DGF,0,18,9


## 6. Verifica geni dall'articolo (Mourão et al. 2016)

L'articolo ha validato con RT-PCR 5 geni selezionati da questo dataset:
- **DEFB1** (defensin beta-1): DOWN in DGF
- **FABP3** (fatty acid binding protein 3): DOWN in DGF (miglior predittore, OR=41.1)
- **GK** (glycerol kinase): DOWN in DGF
- ACSL4: tendenza DOWN ma non significativo nella RT-PCR
- CUBN (cubilin): non significativo nella RT-PCR

Questi sono geni **protettivi/tubulari** la cui perdita di espressione indica danno.
Se i dati sono caricati correttamente, devono essere DOWN in DGF.

In [9]:
# === Verifica geni Mourão 2016 ===
validation_genes = {
    "DEFB1": ("defensin beta-1", "DOWN", True),
    "FABP3": ("fatty acid binding protein 3", "DOWN", True),
    "GK":    ("glycerol kinase", "DOWN", True),
    "ACSL4": ("acyl-CoA synthetase long-chain 4", "DOWN", False),
    "CUBN":  ("cubilin", "DOWN", False),
}

if not tc_to_gene:
    print("⚠️  Annotazione mancante — skip verifica")
else:
    # Inverti mapping gene → transcript cluster IDs
    from collections import defaultdict
    gene_to_tcs = defaultdict(list)
    for tc, g in tc_to_gene.items():
        gene_to_tcs[g].append(tc)
    
    print("VERIFICA GENI MOURÃO 2016 — DGF vs non-DGF")
    print("=" * 75)
    print(f"{'Gene':<8s} {'TC_ID':<12s} {'mean_DGF':>9s} {'mean_noDGF':>11s} {'log2FC':>8s} {'p-val':>10s} {'Atteso':>7s} {'OK':>4s}")
    print("-" * 75)
    
    n_correct = 0
    n_signif = 0
    n_total = 0
    
    for gene, (desc, expected_dir, signif_in_paper) in validation_genes.items():
        tcs = [t for t in gene_to_tcs.get(gene, []) if t in expr_raw.index]
        if not tcs:
            print(f"{gene:<8s} {'NOT FOUND':<12s}")
            continue
        
        # Per ogni TC, calcola DE e prendi quello con p-value migliore
        best_p = 1
        best_row = None
        for tc in tcs:
            d_vals = expr_raw.loc[tc, dgf_samples].dropna().values.astype(float)
            n_vals = expr_raw.loc[tc, nodgf_samples].dropna().values.astype(float)
            if len(d_vals) < 2 or len(n_vals) < 2:
                continue
            t_stat, pval = stats.ttest_ind(d_vals, n_vals, equal_var=False)
            fc = d_vals.mean() - n_vals.mean()
            if pval < best_p:
                best_p = pval
                best_row = (tc, d_vals.mean(), n_vals.mean(), fc, pval)
        
        if best_row:
            tc, m_dgf, m_nodgf, fc, pv = best_row
            n_total += 1
            direction = "DOWN" if fc < 0 else "UP"
            ok = "✅" if direction == expected_dir else "⚠️"
            if direction == expected_dir:
                n_correct += 1
            if pv < 0.05:
                n_signif += 1
            sig_mark = " *" if signif_in_paper else ""
            print(f"{gene:<8s} {tc:<12s} {m_dgf:>9.3f} {m_nodgf:>11.3f} {fc:>+8.3f} {pv:>10.2e} {expected_dir:>7s} {ok:>4s}{sig_mark}")
    
    print("-" * 75)
    print(f"  * = significativo in RT-PCR nell'articolo")
    print(f"\nDirezione corretta: {n_correct}/{n_total}")
    print(f"p < 0.05 (nominale): {n_signif}/{n_total}")
    
    if n_correct >= 3:
        print("\n✅ DATI COERENTI con l'articolo — direzione confermata")
    elif n_correct >= 2:
        print("\n⚠️  Parzialmente coerente — direzione mista")
    else:
        print("\n❌ Direzioni non coerenti — verificare caricamento")

VERIFICA GENI MOURÃO 2016 — DGF vs non-DGF
Gene     TC_ID         mean_DGF  mean_noDGF   log2FC      p-val  Atteso   OK
---------------------------------------------------------------------------
DEFB1    8149097          8.245       8.754   -0.510   3.12e-03    DOWN    ✅ *
FABP3    7914342          9.288       9.680   -0.392   1.25e-02    DOWN    ✅ *
GK       8174103          9.739      10.325   -0.587   7.11e-04    DOWN    ✅ *
ACSL4    8174474          9.546       9.139   +0.408   1.29e-03    DOWN   ⚠️
CUBN     7932326         10.153      11.065   -0.911   3.24e-03    DOWN    ✅
---------------------------------------------------------------------------
  * = significativo in RT-PCR nell'articolo

Direzione corretta: 4/5
p < 0.05 (nominale): 5/5

✅ DATI COERENTI con l'articolo — direzione confermata


## 7. Espressione Differenziale

Welch t-test su tutti i transcript cluster, FDR con Benjamini-Hochberg.
Confronti:
1. **DGF vs non-DGF** (confronto primario per Cat. B)
2. **pDGF vs non-DGF** (sottogruppo severo)

In [10]:
from statsmodels.stats.multitest import multipletests

def de_microarray(expr_df, samples_case, samples_ctrl,
                  comparison_name="", dataset="GSE54888"):
    """DE con Welch t-test su valori log2 (RMA)."""
    case = [s for s in samples_case if s in expr_df.columns]
    ctrl = [s for s in samples_ctrl if s in expr_df.columns]

    print(f"\n{'='*60}")
    print(f"DE: {comparison_name}")
    print(f"  Case: {len(case)}, Control: {len(ctrl)}")

    if len(case) < 2 or len(ctrl) < 2:
        print("  [SKIP]")
        return pd.DataFrame()

    results = []
    for tc in expr_df.index:
        cv = expr_df.loc[tc, case].dropna().values.astype(float)
        tv = expr_df.loc[tc, ctrl].dropna().values.astype(float)
        if len(cv) < 2 or len(tv) < 2:
            continue
        tstat, pval = stats.ttest_ind(cv, tv, equal_var=False)
        results.append({
            "tc_id":          tc,
            "log2FoldChange": float(cv.mean() - tv.mean()),
            "pvalue":         float(pval),
            "t_statistic":    float(tstat),
            "mean_case":      float(cv.mean()),
            "mean_ctrl":      float(tv.mean()),
        })

    df = pd.DataFrame(results).set_index("tc_id")
    _, df["padj"], _, _ = multipletests(df["pvalue"].fillna(1), method="fdr_bh")

    sig = df[(df["padj"] < FDR_THRESHOLD) & (df["log2FoldChange"].abs() > FC_THRESHOLD)]
    print(f"  TC testati:       {len(df)}")
    print(f"  Significativi (|log2FC|>{FC_THRESHOLD}, FDR<{FDR_THRESHOLD}): "
          f"{len(sig)}  ({(sig['log2FoldChange']>0).sum()}↑ "
          f"{(sig['log2FoldChange']<0).sum()}↓)")
    print(f"  (p nominale < 0.05: {(df['pvalue'] < 0.05).sum()})")

    df["comparison"] = comparison_name
    df["dataset"]    = dataset
    return df

In [11]:
# === Esegui DE ===

# Confronto 1 (primario): DGF vs non-DGF
de_dgf = de_microarray(
    expr_raw, dgf_samples, nodgf_samples,
    comparison_name="DGF_vs_noDGF"
)

# Confronto 2: pDGF (>14d) vs non-DGF
de_pdgf = de_microarray(
    expr_raw, pdgf_samples, nodgf_samples,
    comparison_name="pDGF_vs_noDGF"
)


DE: DGF_vs_noDGF
  Case: 27, Control: 27


  TC testati:       33252
  Significativi (|log2FC|>0.379, FDR<0.05): 0  (0↑ 0↓)
  (p nominale < 0.05: 2874)

DE: pDGF_vs_noDGF
  Case: 13, Control: 27
  TC testati:       33252
  Significativi (|log2FC|>0.379, FDR<0.05): 0  (0↑ 0↓)
  (p nominale < 0.05: 3425)


In [12]:
# Ispezione risultati
for name, de_df in [("DGF_vs_noDGF", de_dgf), ("pDGF_vs_noDGF", de_pdgf)]:
    if de_df.empty:
        continue
    print(f"\n--- {name} ---")
    sig = de_df[(de_df["padj"] < FDR_THRESHOLD) & (de_df["log2FoldChange"].abs() > FC_THRESHOLD)]
    print(f"FDR-significativi: {len(sig)}")
    print(f"log2FC: mean={de_df['log2FoldChange'].mean():.4f}, SD={de_df['log2FoldChange'].std():.4f}")
    print(f"\nTop 15 per p-value:")
    display(de_df.sort_values("pvalue").head(15)[
        ["log2FoldChange", "pvalue", "padj", "mean_case", "mean_ctrl"]])


--- DGF_vs_noDGF ---
FDR-significativi: 0
log2FC: mean=-0.0034, SD=0.1102

Top 15 per p-value:


,log2FoldChange,pvalue,padj,mean_case,mean_ctrl
tc_id,,,,,
8139244,-0.201204,0.000046,0.493547,9.592890,9.794095
8104066,-0.427550,0.000086,0.493547,5.750372,6.177922
8159004,0.533769,0.000103,0.493547,6.375707,5.841939
7925028,0.240992,0.000112,0.493547,7.502606,7.261615
8110392,0.245216,0.000137,0.493547,8.931981,8.686765
8039166,0.223049,0.000156,0.493547,9.077307,8.854257
8133219,0.203247,0.000201,0.493547,7.853513,7.650266
8034754,0.157147,0.000202,0.493547,7.046456,6.889310
8138310,-0.451388,0.000203,0.493547,4.910866,5.362254



--- pDGF_vs_noDGF ---
FDR-significativi: 0
log2FC: mean=-0.0057, SD=0.1419

Top 15 per p-value:


,log2FoldChange,pvalue,padj,mean_case,mean_ctrl
tc_id,,,,,
8046060,-0.221123,0.000002,0.058673,2.402119,2.623242
8180361,0.340758,0.000005,0.058673,6.917309,6.576551
8069519,0.278557,0.000005,0.058673,7.150948,6.872391
8149097,-0.754886,0.000013,0.092693,7.999218,8.754104
7929988,0.170433,0.000014,0.092693,6.939769,6.769336
7915170,0.197757,0.000026,0.122068,7.883151,7.685394
7947332,-0.381178,0.000029,0.122068,5.465086,5.846265
8178346,0.214231,0.000030,0.122068,7.751434,7.537203
8124742,0.207443,0.000033,0.122068,7.796423,7.588980


## 8. Transcript Cluster → Gene Symbol

In [13]:
def tc_to_genes(de_df, tc_gene_map):
    """Converte DE tc-level → gene-level (best TC per gene)."""
    if de_df.empty:
        return de_df
    df = de_df.copy()
    df["gene_symbol"] = df.index.astype(str).map(tc_gene_map)
    n0 = len(df)
    df = df.dropna(subset=["gene_symbol"])
    df = df[~df["gene_symbol"].isin(["", "nan", "---", "NA"])]
    n_mapped = len(df)
    df = df.sort_values("pvalue").groupby("gene_symbol").first()
    print(f"  Mapped: {n_mapped}/{n0} TC → {len(df)} geni unici")
    return df

if tc_to_gene:
    print("Mapping DGF_vs_noDGF:")
    de_dgf_genes = tc_to_genes(de_dgf, tc_to_gene)
    print("\nMapping pDGF_vs_noDGF:")
    de_pdgf_genes = tc_to_genes(de_pdgf, tc_to_gene)
else:
    print("⚠️  Annotazione mancante")
    de_dgf_genes = de_pdgf_genes = pd.DataFrame()

Mapping DGF_vs_noDGF:
  Mapped: 25293/33252 TC → 23307 geni unici

Mapping pDGF_vs_noDGF:
  Mapped: 25293/33252 TC → 23307 geni unici


#### Seed genes (sanity check)

In [14]:
if not de_dgf_genes.empty:
    for name, de_g in [("DGF_vs_noDGF", de_dgf_genes), ("pDGF_vs_noDGF", de_pdgf_genes)]:
        seed = de_g[(de_g["padj"] < FDR_THRESHOLD) & (de_g["log2FoldChange"].abs() > FC_THRESHOLD)]
        print(f"\n{name}:")
        print(f"  Seed genes (FDR<0.05, |log2FC|>0.379): {len(seed)}")
        print(f"  Up: {(seed['log2FoldChange']>0).sum()}, Down: {(seed['log2FoldChange']<0).sum()}")
        if len(seed) > 0:
            print(f"  Top 10:")
            for g, r in seed.sort_values("padj").head(10).iterrows():
                print(f"    {g:15s}  FC={r['log2FoldChange']:+.3f}  FDR={r['padj']:.2e}")


DGF_vs_noDGF:
  Seed genes (FDR<0.05, |log2FC|>0.379): 0
  Up: 0, Down: 0

pDGF_vs_noDGF:
  Seed genes (FDR<0.05, |log2FC|>0.379): 0
  Up: 0, Down: 0


## 9. Salvataggio

Output per la Fase 2 (R):
- `de_GSE54888_DGF_vs_IGF.csv` — DGF vs non-DGF (naming coerente pipeline)
- `de_GSE54888_pDGF_vs_noDGF.csv` — pDGF >14d vs non-DGF
- `sample_labels_GSE54888.csv`
- `expr_GSE54888.csv`

In [15]:
if not tc_to_gene:
    print("⚠️  Annotazione mancante — salvataggio TC-level")
    de_dgf.index.name = "tc_id"
    de_dgf[["log2FoldChange", "pvalue", "padj", "comparison", "dataset"]].to_csv(
        OUTPUT_DIR / "de_GSE54888_DGF_vs_IGF_TC_LEVEL.csv")
else:
    # 9a. DE DGF vs non-DGF (primario Cat. B)
    de_out = de_dgf_genes[["log2FoldChange", "pvalue", "padj", "comparison", "dataset"]].copy()
    de_out.index.name = "gene"
    de_out["comparison"] = "DGF_vs_noDGF"
    de_out["dataset"] = "GSE54888"
    de_out.to_csv(OUTPUT_DIR / "de_GSE54888_DGF_vs_IGF.csv")
    print(f"✅ DGF vs noDGF: {len(de_out)} geni → de_GSE54888_DGF_vs_IGF.csv")

    # 9b. DE pDGF vs non-DGF
    de_out2 = de_pdgf_genes[["log2FoldChange", "pvalue", "padj", "comparison", "dataset"]].copy()
    de_out2.index.name = "gene"
    de_out2["comparison"] = "pDGF_vs_noDGF"
    de_out2["dataset"] = "GSE54888"
    de_out2.to_csv(OUTPUT_DIR / "de_GSE54888_pDGF_vs_noDGF.csv")
    print(f"✅ pDGF vs noDGF: {len(de_out2)} geni → de_GSE54888_pDGF_vs_noDGF.csv")

# 9c. Sample labels
all_s = dgf_samples + nodgf_samples
sample_labels = pd.DataFrame({
    "sample": all_s,
    "condition": ["DGF" if s in dgf_samples else "noDGF" for s in all_s],
    "patient": [meta.loc[s, "donor_id"] if s in meta.index else "" for s in all_s],
    "dataset": "GSE54888"
})
sample_labels.to_csv(OUTPUT_DIR / "sample_labels_GSE54888.csv", index=False)
print(f"✅ Labels: {len(sample_labels)} campioni")

# 9d. Matrice espressione gene × sample
if tc_to_gene:
    samples_ok = [s for s in all_s if s in expr_raw.columns]
    best_tc_map = de_dgf.copy()
    best_tc_map["gene_symbol"] = best_tc_map.index.map(tc_to_gene)
    best_tc_map = best_tc_map.dropna(subset=["gene_symbol"])
    best_tc_map = best_tc_map[~best_tc_map["gene_symbol"].isin(["", "nan", "---", "NA"])]
    best_tc_map = best_tc_map.sort_values("pvalue").drop_duplicates(subset="gene_symbol", keep="first")
    gene_to_best_tc = dict(zip(best_tc_map["gene_symbol"], best_tc_map.index))
    tcs_to_keep = list(gene_to_best_tc.values())
    expr_for_r = expr_raw.loc[expr_raw.index.isin(tcs_to_keep), samples_ok].copy()
    expr_for_r.index = expr_for_r.index.map({v: k for k, v in gene_to_best_tc.items()})
    expr_for_r.to_csv(OUTPUT_DIR / "expr_GSE54888.csv")
    print(f"✅ Expr: {expr_for_r.shape[0]} geni × {expr_for_r.shape[1]} campioni")

✅ DGF vs noDGF: 23307 geni → de_GSE54888_DGF_vs_IGF.csv
✅ pDGF vs noDGF: 23307 geni → de_GSE54888_pDGF_vs_noDGF.csv
✅ Labels: 54 campioni
✅ Expr: 23307 geni × 54 campioni


In [16]:
print("\n" + "=" * 60)
print("RIEPILOGO — GSE54888 (Cat. B: Validazione)")
print("=" * 60)
print(f"Campioni:  54 biopsie pre-impianto da DD")
print(f"  DGF: {len(dgf_samples)}, non-DGF: {len(nodgf_samples)}")
print(f"  pDGF (>14d): {len(pdgf_samples)}")
print(f"  eGFR Low: {len(egfr_low)}, High: {len(egfr_high)}")
print(f"Piattaforma: GPL6244 (Affy HuGene-1.0-ST, gene-level, RMA)")
print(f"TC totali: {len(expr_raw)}")
if tc_to_gene:
    print(f"Geni mappati: ~{len(set(tc_to_gene.values()))}")
    sig_d = de_dgf_genes[(de_dgf_genes['padj']<FDR_THRESHOLD) & (de_dgf_genes['log2FoldChange'].abs()>FC_THRESHOLD)]
    sig_p = de_pdgf_genes[(de_pdgf_genes['padj']<FDR_THRESHOLD) & (de_pdgf_genes['log2FoldChange'].abs()>FC_THRESHOLD)]
    print(f"\nDE DGF vs noDGF:  {len(sig_d)} significativi")
    print(f"DE pDGF vs noDGF: {len(sig_p)} significativi")
print(f"\nFile in: {OUTPUT_DIR}/")
print(f"\n📋 Per la validazione in R (Fase 2):")
print(f"   GSEA pre-ranked sui seed genes Cat. A")
print(f"   usando de_GSE54888_DGF_vs_IGF.csv")


RIEPILOGO — GSE54888 (Cat. B: Validazione)
Campioni:  54 biopsie pre-impianto da DD
  DGF: 27, non-DGF: 27
  pDGF (>14d): 13
  eGFR Low: 18, High: 35
Piattaforma: GPL6244 (Affy HuGene-1.0-ST, gene-level, RMA)
TC totali: 33252
Geni mappati: ~23307

DE DGF vs noDGF:  0 significativi
DE pDGF vs noDGF: 0 significativi

File in: /Users/robertocasale/Desktop/lavoro_FBF/magic_solution/output/fase1/GSE54888/

📋 Per la validazione in R (Fase 2):
   GSEA pre-ranked sui seed genes Cat. A
   usando de_GSE54888_DGF_vs_IGF.csv
